In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("LLM_API_KEY"))

resp = client.chat.completions.create(
    model="gpt-5.2",  # <- change to the exact model ID OpenAI provides
    messages=[
        {"role": "user", "content": "Test message"}
    ],
)

print(resp.choices[0].message.content)

In [ ]:
import os
import re

def read_file_clean(location):
    """
    Reads a C source code file, removes comments, newlines, and tabs,
    and returns compact code suitable for functional comparison.

    Parameters:
        location (str): Path to the C source code file.

    Returns:
        str: Cleaned and compact code.

    Raises:
        FileNotFoundError: If the file does not exist.
        PermissionError: If reading is not allowed.
        IOError: For other I/O issues.
    """
    if not os.path.isfile(location):
        raise FileNotFoundError(f"The file at '{location}' does not exist.")

    try:
        with open(location, 'r', encoding='utf-8') as file:
            code = file.read()

        # Remove block comments (/* ... */)
        code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)

        # Remove line comments (// ...)
        code = re.sub(r'//.*', '', code)

        # Remove tabs, newlines, and multiple spaces
        code = code.replace('\n', ' ').replace('\t', ' ')
        code = re.sub(r'\s+', ' ', code).strip()

        return code

    except PermissionError:
        raise PermissionError(f"Permission denied while reading '{location}'.")
    except IOError as e:
        raise IOError(f"An I/O error occurred while reading '{location}': {e}")



In [ ]:
def get_system_prompt() -> str:
    return (
        "You are a helpful and precise coding assistant. "
        "Your job is to identify whether two code snippets behave differently in terms of functionality."
    )


In [ ]:
def get_cot_system_prompt() -> str:
    return (
        "You are an expert in program analysis and formal verification.\n\n"
        "Your task is to determine semantic (observational) equivalence between a source program and a mutant program.\n\n"
        "You must reason about program behavior, not just syntax.\n\n"
        "Definition:\n"
        "Two programs are observationally equivalent if, for every feasible input, their observable outputs are identical.\n\n"
        "Critical requirements:\n"
        "- Do NOT conclude non-equivalence based only on syntactic differences.\n"
        "- You MUST analyze how differences propagate through the program.\n"
        "- You MUST check whether differences are masked before reaching the output.\n"
        "- You MUST consider interaction between multiple bugs.\n\n"
        "You must explicitly reason about the following masking effects:\n"
        "1. Overwrite masking (later assignments remove divergence)\n"
        "2. Control-flow masking (divergent code not executed)\n"
        "3. Output masking (internal difference does not affect output)\n"
        "4. Interaction masking (one bug cancels or neutralizes another)\n"
        "5. Path-specific masking (difference appears only on some paths)\n\n"
        "For each identified difference:\n"
        "- Identify affected variables or predicates\n"
        "- Trace forward propagation\n"
        "- Determine whether divergence survives or is masked\n\n"
        "For multi-bug mutants:\n"
        "- Analyze combined effects, not just each bug independently\n"
        "- Check if bugs amplify, cancel, or hide each other\n\n"
        "Do NOT skip propagation analysis.\n\n"
        "Final decision rules:\n"
        "- Equivalent → no feasible input causes output difference\n"
        "- Non-equivalent → at least one feasible input causes output difference\n"
        "- Conditionally non-equivalent → only some paths diverge\n"
        "- Masked divergence → internal differences exist but do not reach output\n\n"
        "Be precise, structured, and conservative in conclusions.\n"
        "If uncertain, explicitly state limitations."
    )


In [ ]:
def get_user_prompt(source,mutant):
    prompt = f"Here is the original (source) code:\n\n{source}\n\nAnd here is the mutated version of the code:\n\n{mutant}\n\nDo these two pieces of code have any **functional difference**? Respond strictly with \"Yes\" or \"No\" only. Explain your answer in single line if required provide code responsible with line number in mutant, if there are multiple discrepancies each will have one line explanation."
    return prompt

In [ ]:
def get_cot_user_prompt(source: str, mutant: str) -> str:
    return (
        "Task:\n"
        "Compare the following source program (S) and mutant program (M) for observational equivalence.\n\n"
        "Follow the required reasoning steps carefully.\n\n"
        "Steps to perform:\n\n"
        "1. Identify all semantic differences between S and M.\n"
        "2. For each difference:\n"
        "   - List directly affected variables or conditions\n"
        "   - Describe immediate effect\n"
        "3. Trace forward propagation:\n"
        "   - How does this difference influence later computations?\n"
        "   - Does it affect control flow?\n"
        "   - Does it reach output variables?\n"
        "4. Perform masking analysis:\n"
        "   - Is the difference overwritten later?\n"
        "   - Is it blocked by path conditions?\n"
        "   - Is it masked at output?\n"
        "5. Perform interaction analysis:\n"
        "   - Do multiple differences interact?\n"
        "   - Does one difference cancel or amplify another?\n"
        "   - Check explicitly for one-way masking (one bug/change masks or hides the effect of another).\n"
        "   - Check explicitly for two-way masking (two or more bugs/changes cancel each other so that combined behavior appears correct).\n"
        "   - IMPORTANT: If any such masking occurs, you MUST say so explicitly, and identify which bugs/changes are involved.\n"
        "6. Perform path-sensitive reasoning if needed.\n"
        "7. Determine whether any feasible input leads to different outputs.\n\n"
        "Output requirements (STRICT):\n"
        "- Return ONLY valid JSON. No markdown, no code fences, no extra text.\n"
        "- The JSON must include an upfront answer that starts with Yes/No and a one-line justification.\n"
        "- That Yes/No and one-liner MUST explicitly state whether any bug/change is masked by another bug/change (e.g., Yes, outputs differ and no masking between bugs; or No, outputs are equivalent because bug B masks bug A).\n"
        "- The JSON must use these exact top-level keys: answer, one_liner, A_observable_output_variables, B_differences_identified, C_propagation_analysis, D_masking_analysis, E_interaction_analysis, F_surviving_divergences_reaching_output, G_final_verdict, H_witness_input_or_condition, I_confidence_and_limitations.\n"
        "- The meaning of the keys is as follows:\n"
        "  - answer: Yes/No to \"Are S and M observationally equivalent?\"\n"
        "  - one_liner: one sentence, starting with Yes/No, that explicitly mentions whether masking between bugs is present or absent.\n"
        "  - A_observable_output_variables: list of variables considered as observable outputs.\n"
        "  - B_differences_identified: list of semantic differences (with ids, descriptions, directly_affected, immediate_effect).\n"
        "  - C_propagation_analysis: mapping from difference id to how it propagates.\n"
        "  - D_masking_analysis: structured description of overwrite, control_flow, output, interaction, path_specific masking.\n"
        "  - E_interaction_analysis: summary of interactions between multiple differences.\n"
        "  - F_surviving_divergences_reaching_output: list of differences that still reach outputs (with how).\n"
        "  - G_final_verdict: one of Equivalent, Non-equivalent, Conditionally non-equivalent, Masked divergence.\n"
        "  - H_witness_input_or_condition: concrete or symbolic input/condition if non-equivalent.\n"
        "  - I_confidence_and_limitations: object with numeric confidence and textual limitations.\n\n"
        "---\n\n"
        "Source Program (S):\n"
        f"{source}\n\n"
        "---\n\n"
        "Mutant Program (M):\n"
        f"{mutant}"
    )


In [ ]:
from openai import OpenAI
# Create OpenAI client instance
client = OpenAI(api_key=openai.api_key)


In [ ]:
# Updated function for a single message input
def send_prompt(message, model="gpt-4o", temperature=0):
    """
    Send a single user prompt to the OpenAI Chat API with a system message.

    Parameters:
        message (str): The user's prompt.
        model (str): The OpenAI model to use (e.g., 'gpt-4o').
        temperature (float): Sampling temperature for response randomness.

    Returns:
        reply (str): The assistant's response.
        usage (dict): Token usage details.
    """
    messages = [
        {"role": "system", "content": get_system_prompt()},
        {"role": "user", "content": message}
    ]

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature
        )
        reply = response.choices[0].message.content
        usage = response.usage
        return reply, usage
    except Exception as e:
        print(model)
        print("OpenAI API error:", str(e))
        return None, None

In [ ]:
def send_cot_prompt(message, model="gpt-5.2", temperature=0):
    """
    Send a structured CoT-style equivalence prompt to the OpenAI Chat API.

    Args:
        message: Either (source_code, mutant_code) tuple/list OR a dict with keys {'source','mutant'}.
        model: OpenAI model name.
        temperature: Sampling temperature.

    Returns:
        reply (str): Assistant response.
        usage (dict): Token usage details.
    """
    if isinstance(message, (tuple, list)) and len(message) == 2:
        source, mutant = message
    elif isinstance(message, dict) and 'source' in message and 'mutant' in message:
        source, mutant = message['source'], message['mutant']
    else:
        raise ValueError("message must be (source, mutant) or {'source':..., 'mutant':...}")

    user_prompt = get_cot_user_prompt(source, mutant)
    messages = [
        {"role": "system", "content": get_cot_system_prompt()},
        {"role": "user", "content": user_prompt},
    ]

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
        )
        reply = response.choices[0].message.content
        usage = response.usage
        return reply, usage
    except Exception as e:
        print(model)
        print("OpenAI API error:", str(e))
        return None, None


In [ ]:
explaination, usage = send_prompt(get_user_prompt(read_file_clean('path/to/source.c')))


In [ ]:
explaination

In [ ]:
usage

In [ ]:
explaination_11, usage_11 = send_prompt(get_user_prompt(read_file_clean('path/to/source_11.c')))


In [ ]:
explaination_15, usage_15 = send_prompt(get_user_prompt(read_file_clean('path/to/source_15.c')))


In [ ]:
explaination_31, usage_31 = send_prompt(get_user_prompt(read_file_clean('path/to/source_31.c')))


In [ ]:
explaination_32, usage_32 = send_prompt(get_user_prompt(read_file_clean('path/to/source_32.c')))


In [ ]:
explaination_40, usage_40 = send_prompt(get_user_prompt(read_file_clean('path/to/source_40.c')))


In [ ]:
print(explaination)
print('----------')
print(explaination_11)
print('----------')
print(explaination_15)
print('----------')
print(explaination_31)
print('----------')
print(explaination_32)
print('----------')
print(explaination_40)

In [ ]:
source = read_file_clean('path/to/source.c')
mutant = read_file_clean('path/to/mutant.c')

explaination, usage = send_cot_prompt((source, mutant), model="gpt-5.2", temperature=0)

In [ ]:
explaination

In [ ]:
source = read_file_clean('path/to/source.c')
mutant = read_file_clean('path/to/mutant.c')

explaination, usage = send_cot_prompt((source, mutant), model="gpt-5.2", temperature=0)

In [ ]:
explaination

In [ ]:
source = read_file_clean('path/to/source.c')
mutant = read_file_clean('path/to/mutant.c')

explaination, usage = send_cot_prompt((source, mutant), model="gpt-5.2", temperature=0)

In [ ]:
explaination

In [ ]:
source = read_file_clean('path/to/source.c')
mutant = read_file_clean('path/to/mutant.c')

explaination, usage = send_cot_prompt((source, mutant), model="gpt-5.2", temperature=0)

In [ ]:
explaination

In [ ]:
source = read_file_clean('path/to/source.c')
mutant = read_file_clean('path/to/mutant.c')

explaination, usage = send_cot_prompt((source, mutant), model="gpt-5.2", temperature=0)

In [ ]:
explaination

In [ ]:
source = read_file_clean('path/to/source.c')
mutant = read_file_clean('path/to/mutant.c')

explaination, usage = send_cot_prompt((source, mutant), model="gpt-5.2", temperature=0)

In [ ]:
explaination